## **Nivel de sincronización**
Utilizando unos auriculares se reproducirá un fragmento de audio simultáneamente por ambos canales (L/R). En cada simulación se irá aumentando el nivel de desfase entre ambos canales hasta comprobar el límite de percepción del participante.
**Objetivo**: máximo desfase acústicamente perceptible 

| Desfase | 10-500 ms | (9 puntos) |
|---|---|---|

In [1]:
import wave
import pyaudio

CHUNK = 1024
AUDIO_FILE = "lib/Ode to Joy - Beethoven.wav"

Funcion que permite reproducir un fragmento de audio a través de la salida por defecto del equipo, ofreciendo la posibilidad de configurar el balance entre los dos canales stereo.

In [ ]:
global stop_threads
def play_audio(channel_map):
	stream_info = pyaudio.PaMacCoreStreamInfo(
			flags=pyaudio.PaMacCoreStreamInfo.paMacCorePlayNice,
			channel_map=channel_map)

	with wave.open(AUDIO_FILE, 'rb') as wf:
		p = pyaudio.PyAudio()

		stream = p.open(format=p.get_format_from_width(wf.getsampwidth()), channels=wf.getnchannels(), rate=wf.getframerate(), output_host_api_specific_stream_info=stream_info, output=True)
		while len(data := wf.readframes(CHUNK)):
			if stop_threads:
				break
			stream.write(data)

		stream.close()
		p.terminate()


Definición de la lista donde se encuentran los valores empleados en la prueba.

In [8]:
import numpy as np
delay_set = np.linspace(0.01, 0.5, 9)
delay_set

array([0.01   , 0.07125, 0.1325 , 0.19375, 0.255  , 0.31625, 0.3775 ,
       0.43875, 0.5    ])

Ejecución de la prueba con cada uno de los valores definidos anteriormente, de manera secuencial.

In [23]:
from threading import Thread
import time

stop_threads = False

print("> NESCS-01: from 10ms to 500ms (9 steps)")

for DELAY in delay_set:
	print("\t- Playing audio with {:.1f}ms delay...".format(DELAY*1000), end="")
	a = Thread(target=play_audio, args=((1,-1),))
	b = Thread(target=play_audio, args=((-1,1),))

	a.start()
	time.sleep(DELAY)
	b.start()

	try:
		b.join()
		print(" Done!")
	except KeyboardInterrupt:
		stop_threads = True
		print("\n\nEarly stopping...")
		break

> NESCS-01: from 10ms to 500ms (9 steps)
	- Playing audio with 10.0ms delay...

Early stopping...
